# Preparar e auditar reset externo

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="09_reset_total_ambiente_poc"


{'mode': 'READ_EXISTING',
 'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 30,
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': '8a186e1f-3eb9-4819-b6ec-f99ff331853e'}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Selecionar modo e rebootstrap, sem executar exclusão. O controlador do host encerra o kernel em factory-reset; Run All é sempre somente leitura.

In [2]:
from src.workbench.reset_review import review_plan
evidence=review_plan(params)
display(evidence["commands"])
display({"state":evidence["state"],"destructive_execution":"NOT_EXECUTED"})


{'preview': 'python deployment/scripts/reset_poc.py preview --environment poc --project marista --mode reset-data',
 'execution': 'python deployment/scripts/reset_poc.py execute --environment poc --project marista-correcoes-isolated --plan-id 8a186e1f-3eb9-4819-b6ec-f99ff331853e --execute --reinitialize demo-ready',
 'reinitialize': 'demo-ready'}

{'state': 'PLAN_AVAILABLE', 'destructive_execution': 'NOT_EXECUTED'}

2. Inventário, impacto, arquivos preservados e resultado. Plano expirado deve ser recriado no terminal; nunca execute comando antigo sem novo preview.

In [3]:
display(evidence.get("inventory",{}))
display(evidence.get("protected_files",{}))
display(evidence.get("result",{}))


{'bucket': 'pucpr-poc',
 'buckets': ['pucpr-poc'],
 'database_names': ['mlflow_db', 'planejamento_db', 'postgres'],
 'databases': {'mlflow_db': [{'table': 'alembic_version',
    'count': 1,
    'digest': '6928ed6a0b30741f6574f8d7c1d03428'},
   {'table': 'assessments',
    'count': 0,
    'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
   {'table': 'budget_policies',
    'count': 0,
    'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
   {'table': 'datasets',
    'count': 0,
    'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
   {'table': 'endpoint_bindings',
    'count': 0,
    'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
   {'table': 'endpoint_model_mappings',
    'count': 0,
    'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
   {'table': 'endpoint_tags',
    'count': 0,
    'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
   {'table': 'endpoints',
    'count': 0,
    'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
   {'table': 'entity_associations',
    'count': 0,
    'digest': 'd41d8cd9

{'.env': 'db3b5b7502c574fefbbfc8bdfa07d67c928aad848f4df5883e4cab4309aa48f5',
 'notebooks\\00_validacao_ambiente.ipynb': 'cfb263add9201887f9c4f9dddd83c4714b83ef5b52a1db8f8f0f7cd616cd9f14',
 'notebooks\\01_jornada_completa_planilha_turmas.ipynb': 'eeacc6e47d5cc40cc1eace1731e08845012cf99207e08dc9546ea9c0bed2c0f5',
 'notebooks\\02_teste_inferencia_planilha_turmas.ipynb': '9a9265265f0b7cb85fd32db4cb6e49d22e4d11adc23947d6c834170ca2c9ca82',
 'notebooks\\03_exploracao_preparacao_dados.ipynb': '17e5ce6f8cc31d9c6c16fe973942dd88982480396965011b13e1ed863e7d61cc',
 'notebooks\\04_treinamento_individual.ipynb': 'fee632066b9cc02a40d4b168900854de9e313b3b64b4c5421b3ce86bbaa35c59',
 'notebooks\\05_treinamento_campanha.ipynb': '1c346b81261993ab9482507296bbbbeae4e861807494de179f21d74a46f555b8',
 'notebooks\\06_avaliacao_comparativa.ipynb': 'bd544c6f9f6384d27c5f147536595f025497e31d6b1d5ceab35c0174c2a26f0d',
 'notebooks\\07_registry_governanca.ipynb': '4fe24414624ed8c314a82d5ab925859fe635d0736a5cb57adc7b54e

{'plan_id': '8a186e1f-3eb9-4819-b6ec-f99ff331853e',
 'status': 'RESET_PASS',
 'steps': ['writers stopped', 'exclusive volumes removed'],
 'timestamp_utc': '2026-11-04T18:26:05.248229+00:00',
 'after': {'bucket': 'pucpr-poc',
  'buckets': ['pucpr-poc'],
  'database_names': ['mlflow_db', 'planejamento_db', 'postgres'],
  'databases': {'mlflow_db': [{'table': 'alembic_version',
     'count': 1,
     'digest': '6928ed6a0b30741f6574f8d7c1d03428'},
    {'table': 'assessments',
     'count': 0,
     'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
    {'table': 'budget_policies',
     'count': 0,
     'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
    {'table': 'datasets',
     'count': 0,
     'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
    {'table': 'endpoint_bindings',
     'count': 0,
     'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
    {'table': 'endpoint_model_mappings',
     'count': 0,
     'digest': 'd41d8cd98f00b204e9800998ecf8427e'},
    {'table': 'endpoint_tags',
     'count':

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [4]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v2/09_reset_total_ambiente_poc.json'